In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import joblib

# Load datasets
df_red = pd.read_csv('winequality-red.csv', sep=';')
df_white = pd.read_csv('winequality-white.csv', sep=';')

# Combine datasets (no 'type' feature added)
df = pd.concat([df_red, df_white], axis=0)

# Features and target (drop 'type' by not including it)
X = df.drop('quality', axis=1)
y = df['quality']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler for inference
joblib.dump(scaler, 'scaler.pkl')

# Define model with class_weight='balanced'
model = RandomForestClassifier(class_weight='balanced', random_state=42)

# Define parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [180],
    'max_depth': [None],
    'min_samples_split': [2],
    'min_samples_leaf': [1]
}

# Perform GridSearchCV with cv=4
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=4, n_jobs=-1, scoring='accuracy')
grid_search.fit(X_train_scaled, y_train)

# Best model
best_model = grid_search.best_estimator_
print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Score: {grid_search.best_score_}')

# Evaluate on test set
predictions = best_model.predict(X_test_scaled)
print(f'Test Accuracy: {accuracy_score(y_test, predictions)}')

# Print feature importance
feature_names = X.columns
importances = best_model.feature_importances_
for name, importance in zip(feature_names, importances):
    print(f'Feature: {name}, Importance: {importance:.4f}')

# Save model
joblib.dump(best_model, 'wine_model.pkl')

Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 180}
Best CV Score: 0.6696201219873275
Test Accuracy: 0.6915384615384615
Feature: fixed acidity, Importance: 0.0840
Feature: volatile acidity, Importance: 0.0864
Feature: citric acid, Importance: 0.0721
Feature: residual sugar, Importance: 0.0796
Feature: chlorides, Importance: 0.1046
Feature: free sulfur dioxide, Importance: 0.0936
Feature: total sulfur dioxide, Importance: 0.0886
Feature: density, Importance: 0.1100
Feature: pH, Importance: 0.0862
Feature: sulphates, Importance: 0.0712
Feature: alcohol, Importance: 0.1236


['wine_model.pkl']

In [41]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
import joblib
import numpy as np

# Load datasets
df_red = pd.read_csv('winequality-red.csv', sep=';')
df_white = pd.read_csv('winequality-white.csv', sep=';')

# Combine datasets
df = pd.concat([df_red, df_white], axis=0)

# Features and target
X = df.drop('quality', axis=1)
y = df['quality']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler for inference
joblib.dump(scaler, 'scaler.pkl')

# Define model (Ridge regression for regularization)
model = Ridge()

# Define parameter grid for alpha
param_grid = {'alpha': [36.6]}

# Perform GridSearchCV with cv=4
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=4, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)

# Best model
best_model = grid_search.best_estimator_
print(f'Best Parameters: {grid_search.best_params_}')
print(f'Best CV Score (Negative MSE): {grid_search.best_score_}')

# Evaluate on test set
predictions = best_model.predict(X_test_scaled)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
# Round predictions for classification-like accuracy
predictions_rounded = np.round(predictions).astype(int)
predictions_rounded = np.clip(predictions_rounded, 0, 10)  # Ensure predictions stay in 0-10
accuracy = accuracy_score(y_test, predictions_rounded)

print(f'Test MSE: {mse:.4f}')
print(f'Test R²: {r2:.4f}')
print(f'Test Accuracy (Rounded): {accuracy:.4f}')

# Save model
joblib.dump(best_model, 'wine_model.pkl')

Best Parameters: {'alpha': 36.6}
Best CV Score (Negative MSE): -0.5415985024019405
Test MSE: 0.5458
Test R²: 0.2609
Test Accuracy (Rounded): 0.5408


['wine_model.pkl']